# Backend Comparison: Regression vs PMM

This notebook provides a systematic comparison of SMLR's two emulation backends:

| Backend | Method | Best For |
|---------|--------|----------|
| **Regression** | Fit Lorentzian poles → regress pole parameters | Smooth parameter dependence, fast training |
| **PMM** | Parametric matrix model with learned response matrix | Physics-based extrapolation, sum rule preservation |

## Comparison Metrics

We evaluate backends on:

1. **Prediction Accuracy**: L2 error on interpolation points
2. **Extrapolation Behavior**: Error outside training parameter range
3. **Parameter Efficiency**: Performance vs number of model parameters
4. **Training Time**: Wall-clock time to fit
5. **Sum Rule Preservation**: Conservation of integrated spectral weight
6. **Robustness**: Sensitivity to hyperparameters


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import time
from smlr import Surrogate, StrengthDataset, StrengthSample
from smlr.metrics import normalized_l2


## Test Problem Setup

We use a synthetic strength function with known analytic form, allowing 
exact ground truth comparisons. The spectrum has multiple peaks whose 
positions, widths, and amplitudes all depend on input parameters.


In [ ]:
def strength_function(params, energy):
    """Multi-peak strength function with parameter-dependent features.
    
    Parameters control:
    - params[0]: Controls peak 1 position and peak 2 amplitude
    - params[1]: Controls peak 2 position and global width
    """
    p1, p2 = params
    
    # Peak 1: Low energy
    e1 = 5 + 4 * p1
    s1 = 0.8
    
    # Peak 2: High energy, amplitude depends on p1
    e2 = 15 + 5 * p2
    s2 = 0.6 + 0.4 * p1
    
    # Peak 3: Small satellite
    e3 = 25 + 2 * p2
    s3 = 0.2
    
    # Width depends on p2
    gamma = 1.5 + 0.8 * p2
    
    def lorentzian(e, e0, g, s):
        return s * g / np.pi / ((e - e0)**2 + g**2)
    
    return (lorentzian(energy, e1, gamma, s1) +
            lorentzian(energy, e2, gamma, s2) +
            lorentzian(energy, e3, gamma, s3))

# Energy grid
energy = np.linspace(0, 40, 250)

# Training data: 6x6 grid in [0,1]^2
train_p1 = np.linspace(0, 1, 6)
train_p2 = np.linspace(0, 1, 6)

train_samples = []
for p1 in train_p1:
    for p2 in train_p2:
        params = np.array([p1, p2])
        spectrum = strength_function(params, energy)
        train_samples.append(StrengthSample(params, energy, spectrum))

train_dataset = StrengthDataset(train_samples)
print(f"Training samples: {len(train_samples)}")

# Test data: 10 random interpolation points
rng = np.random.default_rng(42)
test_interp_params = rng.uniform(0.1, 0.9, size=(10, 2))

# Extrapolation test: points outside training range
test_extrap_params = np.array([
    [1.1, 0.5],   # p1 extrapolated
    [0.5, 1.1],   # p2 extrapolated
    [1.1, 1.1],   # Both extrapolated
    [-0.1, 0.5],  # Negative p1
    [0.5, -0.1],  # Negative p2
])

print(f"Interpolation test points: {len(test_interp_params)}")
print(f"Extrapolation test points: {len(test_extrap_params)}")


## 1. Interpolation Accuracy

First, we compare prediction accuracy on held-out interpolation points.

In [ ]:
def evaluate_interpolation(model, test_params, energy):
    """Compute errors on test points."""
    errors = []
    for params in test_params:
        result = model.predict(params, energy)
        truth = strength_function(params, energy)
        errors.append(normalized_l2(result.spectrum, truth, energy))
    return np.array(errors)

# Configure models to compare
configs = {
    'Regression (n=3)': {'backend': 'regression', 'n_components': 3, 'width_mode': 'global', 'random_state': 42},
    'Regression (n=5)': {'backend': 'regression', 'n_components': 5, 'width_mode': 'global', 'random_state': 42},
    'Regression (n=7)': {'backend': 'regression', 'n_components': 7, 'width_mode': 'global', 'random_state': 42},
    'PMM (poles=5)': {'backend': 'pmm', 'n_poles': 5, 'retain': 0.8},
    'PMM (poles=7)': {'backend': 'pmm', 'n_poles': 7, 'retain': 0.8},
    'PMM (poles=10)': {'backend': 'pmm', 'n_poles': 10, 'retain': 0.8},
}

results = {}
for name, cfg in configs.items():
    backend = cfg.pop('backend')
    
    # Time training
    t_start = time.time()
    model = Surrogate(backend, **cfg)
    model.fit(train_dataset)
    train_time = time.time() - t_start
    
    # Evaluate
    interp_errors = evaluate_interpolation(model, test_interp_params, energy)
    
    results[name] = {
        'model': model,
        'train_time': train_time,
        'interp_mean': np.mean(interp_errors),
        'interp_max': np.max(interp_errors),
        'interp_errors': interp_errors,
    }
    
    print(f"{name:20s}: mean={np.mean(interp_errors):.4f}, max={np.max(interp_errors):.4f}, time={train_time:.3f}s")


In [ ]:
# Visualize interpolation comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Box plot of errors
ax = axes[0]
reg_errors = [results[k]['interp_errors'] for k in results if 'Regression' in k]
pmm_errors = [results[k]['interp_errors'] for k in results if 'PMM' in k]
reg_labels = [k for k in results if 'Regression' in k]
pmm_labels = [k for k in results if 'PMM' in k]

positions_reg = np.arange(len(reg_errors)) * 2
positions_pmm = np.arange(len(pmm_errors)) * 2 + 0.8

bp1 = ax.boxplot(reg_errors, positions=positions_reg, widths=0.6, patch_artist=True)
bp2 = ax.boxplot(pmm_errors, positions=positions_pmm, widths=0.6, patch_artist=True)

for patch in bp1['boxes']:
    patch.set_facecolor('lightblue')
for patch in bp2['boxes']:
    patch.set_facecolor('lightcoral')

ax.set_xticks((positions_reg + positions_pmm) / 2)
ax.set_xticklabels(['Low', 'Med', 'High'])
ax.set_xlabel('Model Complexity')
ax.set_ylabel('Normalized L2 Error')
ax.set_title('Interpolation Error Distribution')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['Regression', 'PMM'])

# Right: Training time comparison
ax = axes[1]
names = list(results.keys())
times = [results[k]['train_time'] for k in names]
colors = ['steelblue' if 'Regression' in k else 'coral' for k in names]
ax.barh(names, times, color=colors)
ax.set_xlabel('Training Time (s)')
ax.set_title('Training Time Comparison')

plt.tight_layout()
fig


## 2. Extrapolation Behavior

A key difference between backends is their behavior outside the training 
parameter range. PMM's physics-based structure often provides more stable
extrapolation.


In [ ]:
# Pick best of each type based on interpolation
best_reg = 'Regression (n=5)'
best_pmm = 'PMM (poles=7)'

# Evaluate extrapolation
print("Extrapolation Errors:")
print("-" * 50)
for name in [best_reg, best_pmm]:
    model = results[name]['model']
    extrap_errors = []
    for params in test_extrap_params:
        try:
            result = model.predict(params, energy)
            truth = strength_function(params, energy)
            err = normalized_l2(result.spectrum, truth, energy)
        except:
            err = np.nan
        extrap_errors.append(err)
    
    results[name]['extrap_errors'] = np.array(extrap_errors)
    print(f"{name}: {extrap_errors}")


In [ ]:
# Visualize extrapolation at one point
extrap_point = np.array([1.1, 0.5])
truth = strength_function(extrap_point, energy)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name in zip(axes, [best_reg, best_pmm]):
    model = results[name]['model']
    try:
        result = model.predict(extrap_point, energy)
        pred = result.spectrum
        err = normalized_l2(pred, truth, energy)
        
        ax.plot(energy, truth, 'k-', lw=2, label='Ground truth')
        ax.plot(energy, pred, 'r--', lw=2, label=f'{name}')
        ax.set_title(f'{name}\nExtrap. error = {err:.4f}')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {e}', transform=ax.transAxes)
        ax.set_title(f'{name}\nFailed')
    
    ax.set_xlabel('Energy')
    ax.set_ylabel('Strength')
    ax.legend()
    ax.set_xlim(0, 40)

plt.tight_layout()
fig


## 3. Sum Rule Preservation

For physical strength functions, the integrated spectral weight (sum rule) 
is often a conserved quantity. We check how well each backend preserves this.


In [ ]:
def compute_sum_rule(spectrum, energy):
    """Compute integrated spectral weight."""
    return np.trapezoid(spectrum, energy)

# Compute sum rules for training data
train_sum_rules = [compute_sum_rule(s.strength, s.energy) for s in train_dataset.samples]
mean_sum_rule = np.mean(train_sum_rules)
print(f"Training data mean sum rule: {mean_sum_rule:.4f}")

# Check sum rule preservation on test points
fig, ax = plt.subplots(figsize=(8, 5))

for name in [best_reg, best_pmm]:
    model = results[name]['model']
    predicted_srs = []
    true_srs = []
    
    for params in test_interp_params:
        result = model.predict(params, energy)
        truth = strength_function(params, energy)
        
        predicted_srs.append(compute_sum_rule(result.spectrum, energy))
        true_srs.append(compute_sum_rule(truth, energy))
    
    relative_error = (np.array(predicted_srs) - np.array(true_srs)) / np.array(true_srs) * 100
    
    results[name]['sum_rule_error'] = relative_error
    ax.scatter(range(len(relative_error)), relative_error, label=name, s=100, alpha=0.7)

ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Test point index')
ax.set_ylabel('Sum rule error (%)')
ax.set_title('Sum Rule Preservation')
ax.legend()
ax.set_ylim(-10, 10)
fig


## 4. Parameter Efficiency

How many model parameters are needed to achieve a given accuracy level?
This matters for storage and computational efficiency.


In [ ]:
# Estimate parameter counts
def estimate_params(backend, config, n_train, n_dim=2):
    """Rough estimate of effective parameters."""
    if backend == 'regression':
        nc = config.get('n_components', 5)
        # Poles + strengths + widths, each regressed from params
        return nc * 3 * (n_dim + 1)  # Linear regression coefficients
    else:  # pmm
        np_ = config.get('n_poles', 10)
        # Matrix elements scale as n_poles^2 * n_dim
        return np_**2 + np_ * n_dim

# Summary plot
fig, ax = plt.subplots(figsize=(8, 5))

markers = {'Regression': 'o', 'PMM': 's'}
colors_backend = {'Regression': 'steelblue', 'PMM': 'coral'}

for name, res in results.items():
    backend = 'Regression' if 'Regression' in name else 'PMM'
    config = configs[name] if name in configs else {}
    n_params = estimate_params(backend.lower(), config, len(train_samples))
    
    ax.scatter(n_params, res['interp_mean'], 
               marker=markers[backend], c=colors_backend[backend],
               s=150, alpha=0.8, label=name if 'n=3' in name or 'poles=5' in name else '')
    ax.annotate(name.split('(')[1].rstrip(')'), 
                (n_params, res['interp_mean']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_xlabel('Estimated Number of Parameters')
ax.set_ylabel('Mean Interpolation Error')
ax.set_title('Parameter Efficiency')
ax.legend(['Regression', 'PMM'], loc='upper right')
ax.set_yscale('log')
fig


## Summary and Recommendations

### When to use Regression Backend:
- ✅ Smooth, well-sampled parameter space
- ✅ Fast training is essential
- ✅ Interpolation-only use case
- ✅ Simple spectral structure

### When to use PMM Backend:
- ✅ Need extrapolation capability
- ✅ Sum rule preservation matters
- ✅ Physics-based constraints desired
- ✅ Complex spectral evolution
- ✅ Fewer training samples available


In [ ]:
# Generate summary table
import pandas as pd

summary_data = []
for name in [best_reg, best_pmm]:
    res = results[name]
    backend = 'Regression' if 'Regression' in name else 'PMM'
    
    summary_data.append({
        'Backend': backend,
        'Configuration': name.split('(')[1].rstrip(')'),
        'Mean Interp. Error': f"{res['interp_mean']:.4f}",
        'Max Interp. Error': f"{res['interp_max']:.4f}",
        'Training Time (s)': f"{res['train_time']:.3f}",
        'Sum Rule RMSE (%)': f"{np.sqrt(np.mean(res['sum_rule_error']**2)):.2f}",
    })

print("=" * 70)
print("BACKEND COMPARISON SUMMARY")
print("=" * 70)
for d in summary_data:
    print(f"\n{d['Backend']} ({d['Configuration']}):")
    for k, v in d.items():
        if k not in ['Backend', 'Configuration']:
            print(f"  {k}: {v}")
